In [9]:
import pandas as pd
import re

In [10]:
df = pd.read_csv('Complaints.csv')
orders_df = pd.read_csv('Orders.csv')

df['complaint_date'] = pd.to_datetime(df['complaint_date'], errors='coerce')
df = df.sort_values(by=['customer_id', 'complaint_date']).reset_index(drop=True)
df['prior_complaints'] = df.groupby('customer_id').cumcount()
order_counts = orders_df.groupby('customer_id').size().to_dict()
df['total_orders'] = df['customer_id'].map(order_counts).fillna(0)

df['complaint_ratio'] = (df['prior_complaints'] + 1) / df['total_orders'].apply(lambda x: 1 if x == 0 else x)

In [11]:
def custom_pos_tag(text):
    text = str(text).lower()
    words = re.findall(r'\b[a-z]+\b', text)
    
    nouns = {'phone', 'battery', 'screen', 'display', 'laptop', 'charger', 'device', 'headphones', 'earbuds', 'order', 'product', 'box', 'color', 'model'}
    verbs = {'stopped', 'working', 'broke', 'charge', 'restart', 'died', 'ordered', 'got', 'looks', 'returning', 'changed'}
    adjectives = {'dead', 'black', 'hot', 'slow', 'broken', 'bad', 'warm', 'random', 'scratches', 'tampered', 'older', 'different', 'cheap', 'wrong', 'flickering', 'laggy', 'unstable'}
    
    tagged = []
    for w in words:
        if w in nouns:
            tagged.append((w, 'NN'))
        elif w in verbs:
            tagged.append((w, 'VB'))
        elif w in adjectives:
            tagged.append((w, 'JJ'))
        elif w.endswith('ing') or w.endswith('ed'):
            tagged.append((w, 'VB'))
        elif w.endswith('ly'):
            tagged.append((w, 'RB'))
        else:
            tagged.append((w, 'UNK'))
    return tagged

def custom_dependency_parse(tagged_tokens):
    dependencies = []
    n = len(tagged_tokens)
    for i in range(n - 1):
        w1, tag1 = tagged_tokens[i]
        w2, tag2 = tagged_tokens[i+1]
        
        if tag1 == 'JJ' and tag2 == 'NN':
            dependencies.append(('amod', w1, w2))
        elif tag1 == 'NN' and tag2 == 'VB':
            dependencies.append(('nsubj', w1, w2))
        elif tag1 == 'VB' and tag2 == 'JJ':
            dependencies.append(('acomp', w1, w2))
    return dependencies


### Issue Type, Severity, Sentiment, Urgency

In [12]:
def get_issue_type(text):
    text = str(text).lower()
    if any(word in text for word in ['battery', 'charge', 'charging', 'drain']):
        return 'Battery issue'
    elif any(word in text for word in ['screen', 'display', 'monitor', 'blank', 'flicker', 'lines', 'ghost']):
        return 'Screen damage'
    elif any(word in text for word in ['dead', 'stopped working', 'not turn', 'not power', 'not working', 'stuck', 'restarting', 'boot']):
        return 'Device not working'
    elif any(word in text for word in ['heat', 'hot', 'overheat', 'burnt', 'burn', 'warm']):
        return 'Heating issue'
    elif any(word in text for word in ['lag', 'slow', 'hang', 'freeze', 'performance', 'delay']):
        return 'Performance issue'
    return 'Other'

def get_issue_severity(text):
    text = str(text).lower()
    if any(word in text for word in ['broken', 'dead', 'exploded', 'burnt', 'completely']):
        return 'High'
    elif any(word in text for word in ['restarting', 'unstable', 'disconnect', 'loose']):
        return 'Medium'
    elif any(word in text for word in ['slow', 'minor', 'slight', 'laggy', 'flicker']):
        return 'Low'
    return 'Low' 

def get_sentiment(text):
    text = str(text).lower()

    negated_positives = ['not good', 'not great', 'not happy', 'not amazing', 'not perfect', 'not nice']
    for phrase in negated_positives:
        if phrase in text:
            return 'Negative'

    positive_words = {'good', 'great', 'excellent', 'amazing', 'love', 'best', 'ok', 'okay', 'nice', 'awesome', 'perfect', 'happy'}
    negative_words = {'bad', 'terrible', 'worst', 'awful', 'hate', 'cheap', 'poor', 'useless', 'disappointed', 'broke', 'dead', 'meh'}
    
    words = re.findall(r'\b[a-z]+\b', text)
    pos_score = sum(1 for w in words if w in positive_words)
    neg_score = sum(1 for w in words if w in negative_words)
    
    if pos_score > neg_score:
        return 'Positive'
    elif neg_score > pos_score:
        return 'Negative'
    return 'Neutral'

def get_urgency(text):
    text = str(text).lower()
    
    if re.search(r'!!!|\?\?\?', text):
        return 'High'

    urgent_phrases = ['right now', 'as soon as possible', 'at once']
    if any(phrase in text for phrase in urgent_phrases):
        return 'High'
        
    urgent_keywords = {'asap', 'urgent', 'immediately', 'now', 'fast', 'quick'}
    tokens = re.findall(r'\b[a-z]+\b', text)
    if any(token in urgent_keywords for token in tokens):
        return 'High'
    
    return 'Normal'

### 2. Rule-Based Fraud Detection 

In [13]:
def compute_fraud_features(row):
    text = str(row['complaint_text']).lower()
    score = 6
    reason = "Valid defect"
    classification = "Legitimate"
    refund = "Yes"
    
    if 'box' in text and any(w in text for w in ['random', 'broken', 'scratches', 'tampered', 'older', 'different']):
        score, classification, refund, reason = 95, 'Fraud', 'No', 'Switcheroo suspected'
        
    elif 'ordered' in text and ('got' in text or 'looks' in text or 'why' in text):
        score, classification, refund, reason = 93, 'Fraud', 'No', 'Color contradiction'

    elif 'used' in text and ('already' in text or 'returning' in text or 'few' in text or 'event' in text or 'now' in text):
        score, classification, refund, reason = 85, 'Fraud', 'No', 'Usage abuse'

    elif any(phrase in text for phrase in ['wrong item', 'wrong model', 'different from', 'not matching', 'match description']): 
        score, classification, refund, reason = 90, 'Fraud', 'No', 'Unverified mismatch'
        
    elif any(phrase in text for phrase in ['changed my mind', 'dont like', "don't like", 'prefer', 'not my style', 'expected better', 'not what i expected', 'feels cheap', 'not worth']):
        score, classification, refund, reason = 18, 'Fraud', 'No', 'Change of mind'

    else:
        words = text.split()
        if text.strip() in ['ok', 'bad product', 'not good', 'worst', 'bad', 'meh'] or (len(words) <= 3 and 'dead' not in text and 'not working' not in text):
            score, classification, refund, reason = 22, 'Fraud', 'No', 'Vague complaint'

    history_penalty = 0
    prior_complaints = row['prior_complaints']
    ratio = row['complaint_ratio']
    
    if prior_complaints >= 3:
        history_penalty += 15
        classification = "Fraud"
        refund = "No"
        reason = "Frequent Abuser"
        
    if ratio > 0.5 and row['total_orders'] > 2:
        history_penalty += 10
        if classification == "Legitimate":
            classification = "Fraud"
            refund = "No"
            reason = "High Ratio Offender"

    if ratio <= 0.1 and prior_complaints == 0:
        history_penalty -= 5  
        
    score += history_penalty
    
    score = max(0, min(100, score))
    if score >= 80:
        classification = "Fraud"
        refund = "No"
        
    return pd.Series([score, classification, refund, reason])

In [14]:
df['Issue_Type'] = df['complaint_text'].apply(get_issue_type)
df['Issue_Severity'] = df['complaint_text'].apply(get_issue_severity)
df['Sentiment'] = df['complaint_text'].apply(get_sentiment)
df['Urgency'] = df['complaint_text'].apply(get_urgency)

df[['Computed_Fraud_Score', 'Computed_Fraud_Class', 'Computed_Refund_Applicable', 'Computed_Fraud_Reason']] = df.apply(compute_fraud_features, axis=1)

In [15]:
display(df[['customer_id', 'prior_complaints', 'complaint_text', 'Computed_Fraud_Score', 'Computed_Fraud_Class', 'Computed_Fraud_Reason']].head(10))

,customer_id,prior_complaints,complaint_text,Computed_Fraud_Score,Computed_Fraud_Class,Computed_Fraud_Reason
0,C-001,0,My Sony headphones stopped working completely ...,6,Legitimate,Valid defect
1,C-001,1,earbuds not working properly,6,Legitimate,Valid defect
2,C-001,2,earbuds ok but meh,6,Legitimate,Valid defect
3,C-001,3,feels cheap compared to pictures,33,Fraud,Frequent Abuser
4,C-001,4,this isnt what i saw online,31,Fraud,Frequent Abuser
5,C-002,0,Laptop screen is completely black not turning ...,6,Legitimate,Valid defect
6,C-002,1,phone dead after update,6,Legitimate,Valid defect
7,C-002,2,i dont like this color anymore,18,Fraud,Change of mind
8,C-002,3,this is not exactly what i thought it would be,21,Fraud,Frequent Abuser
9,C-002,4,not satisfied overall,47,Fraud,Frequent Abuser
